# Dual ping latency (OKX vs Bybit)

Источник: `output/ping_dual_2h.log` (скачан с VPS `/var/log/spread/ping_dual_2h.log`).

`latency_ms` = локальное время − exchange timestamp (возраст котировки, **не ICMP**).

Прогон: start → stop_requested(signal) → finished; полный 2h не дошёл.

In [ ]:
from pathlib import Path
import re
from datetime import datetime, timezone
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

ROOT = Path("..").resolve()
LOG = ROOT / "output" / "ping_dual_2h.log"
PNG = ROOT / "output" / "ping_dual_2h_latency.png"

ts_re = re.compile(r"^(\d{4}-\d{2}-\d{2}T[^\t]+)\t(okx|bybit)\t(.*)$")
kv_re = re.compile(r"(\w+)=([^\t]+)")

rows = {"okx": [], "bybit": []}
events = []
with LOG.open() as f:
    for line in f:
        line = line.rstrip("\n")
        m = ts_re.match(line)
        if not m:
            if "\tmeta\t" in line:
                parts = line.split("\t")
                if len(parts) >= 3:
                    kvs = dict(kv_re.findall("\t".join(parts[2:])))
                    if "event" in kvs:
                        events.append((parts[0], kvs["event"], kvs))
            continue
        ts_s, exch, rest = m.groups()
        kvs = dict(kv_re.findall(rest))
        if "latency_ms" not in kvs:
            continue
        lat = float(kvs["latency_ms"])
        ts = datetime.fromisoformat(ts_s.replace("Z", "+00:00"))
        age_cts = None
        if kvs.get("age_cts_ms") not in (None, "None", ""):
            try:
                age_cts = float(kvs["age_cts_ms"])
            except ValueError:
                pass
        rows[exch].append((ts, lat, age_cts))

def series(ex):
    data = rows[ex]
    return [d[0] for d in data], np.array([d[1] for d in data], float)

stats = {}
for ex in ("okx", "bybit"):
    _, lat = series(ex)
    stats[ex] = dict(
        n=int(lat.size),
        p50=float(np.percentile(lat, 50)),
        p95=float(np.percentile(lat, 95)),
        p99=float(np.percentile(lat, 99)),
        mean=float(lat.mean()),
        max=float(lat.max()),
        min=float(lat.min()),
    )

print("События:")
for e in events:
    if e[1] in ("start", "stop_requested", "finished", "note"):
        extra = {k: e[2][k] for k in e[2] if k != "event"}
        print(" ", e[0], e[1], extra)
print("\nСводка latency_ms:")
for ex, s in stats.items():
    print(f"  {ex.upper():5} n={s['n']:6d}  p50={s['p50']:6.1f}  p95={s['p95']:6.1f}  p99={s['p99']:6.1f}  mean={s['mean']:6.1f}  max={s['max']:7.1f}  min={s['min']:5.1f}")



In [ ]:
colors = {"okx": "#1f77b4", "bybit": "#ff7f0e"}
labels = {"okx": "OKX", "bybit": "Bybit"}

fig, axes = plt.subplots(
    3, 1, figsize=(12, 9),
    gridspec_kw={"height_ratios": [2.2, 1.4, 1.2]},
)

ax = axes[0]
for ex in ("okx", "bybit"):
    t_py, lat = series(ex)
    step = max(1, len(lat) // 8000)
    ax.plot(t_py[::step], lat[::step], ".", ms=1.5, alpha=0.35, color=colors[ex], label=labels[ex])
ax.set_ylabel("latency_ms")
ax.set_title("Возраст котировок (latency_ms = local − exchange_ts), dual ping")
ax.legend(loc="upper right", markerscale=6)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, min(500, max(stats["okx"]["p99"], stats["bybit"]["p99"]) * 3))
for ts_s, ev, _ in events:
    if ev in ("stop_requested", "finished", "start"):
        t0 = datetime.fromisoformat(ts_s.replace("Z", "+00:00"))
        ax.axvline(t0, color="gray", ls="--" if ev != "start" else ":", alpha=0.6, lw=1)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M", tz=timezone.utc))

ax2 = axes[1]
for ex in ("okx", "bybit"):
    t_py, lat = series(ex)
    t0 = t_py[0].timestamp()
    sec = np.array([t.timestamp() - t0 for t in t_py])
    bins = np.arange(0, sec.max() + 30, 30)
    centers, p50s, p95s = [], [], []
    for i in range(len(bins) - 1):
        mask = (sec >= bins[i]) & (sec < bins[i + 1])
        if mask.sum() < 5:
            continue
        chunk = lat[mask]
        centers.append(t0 + (bins[i] + bins[i + 1]) / 2)
        p50s.append(np.percentile(chunk, 50))
        p95s.append(np.percentile(chunk, 95))
    t_cent = [datetime.fromtimestamp(c, tz=timezone.utc) for c in centers]
    ax2.plot(t_cent, p50s, "-", color=colors[ex], lw=1.5, label=f"{labels[ex]} p50")
    ax2.plot(t_cent, p95s, "--", color=colors[ex], lw=1.2, alpha=0.85, label=f"{labels[ex]} p95")
ax2.set_ylabel("latency_ms")
ax2.set_title("Скользящие p50 / p95 (окна 30 с)")
ax2.legend(loc="upper right", ncol=2, fontsize=8)
ax2.grid(True, alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M", tz=timezone.utc))

ax3 = axes[2]
bins_h = np.linspace(0, max(stats["okx"]["p99"], stats["bybit"]["p99"]) * 1.5, 80)
for ex in ("okx", "bybit"):
    _, lat = series(ex)
    ax3.hist(lat, bins=bins_h, alpha=0.45, color=colors[ex], label=labels[ex], density=True)
ax3.set_xlabel("latency_ms")
ax3.set_ylabel("плотность")
ax3.set_title("Распределение latency_ms")
ax3.legend()
ax3.grid(True, alpha=0.3)

fig.text(
    0.5, 0.01,
    "Прогон оборван signal (~1ч17м из 2ч); не ICMP, а age market data",
    ha="center", fontsize=9, color="#444",
)
fig.tight_layout(rect=[0, 0.03, 1, 1])
PNG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(PNG, dpi=140)
print("Сохранено:", PNG)
plt.show()